In [106]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [107]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [108]:
documents = ["https://github.com/DataTalksClub/llm-zoomcamp/blob/main/01-agentic-rag/lessons/01-intro.md", 
             "https://github.com/DataTalksClub/llm-zoomcamp/blob/main/01-agentic-rag/lessons/02-environment.md", 
             "https://github.com/DataTalksClub/llm-zoomcamp/blob/main/01-agentic-rag/lessons/03-rag.md"
            ]

In [109]:
documents_llm = []

for doc in documents:
        documents_llm.append(doc)

len(documents_llm)

3

In [110]:
documents = documents_llm

In [111]:
doc = documents[0]
print(doc)
print(documents)

https://github.com/DataTalksClub/llm-zoomcamp/blob/main/01-agentic-rag/lessons/01-intro.md
['https://github.com/DataTalksClub/llm-zoomcamp/blob/main/01-agentic-rag/lessons/01-intro.md', 'https://github.com/DataTalksClub/llm-zoomcamp/blob/main/01-agentic-rag/lessons/02-environment.md', 'https://github.com/DataTalksClub/llm-zoomcamp/blob/main/01-agentic-rag/lessons/03-rag.md']


In [112]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [113]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [114]:
import json

user_prompt = json.dumps(documents)

In [115]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [116]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [117]:
result = response.output_parsed

print(result)

questions=['What is the main idea behind the agentic RAG approach, and why would you use an agent instead of a plain retrieval pipeline?', 'What kind of setup do I need before starting the agentic RAG lessons, and what tools or accounts are required?', 'How does the basic RAG workflow turn a user question into an answer, from searching documents to generating the final response?', 'Why can regular RAG break down on harder questions, and what role does planning or tool use play in fixing that?', 'What are the key steps in building a simple RAG system for this course, from preparing the documents to using them in prompts?']


In [118]:
from evaluation_utils import llm_structured

In [119]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['What’s the main difference between a basic RAG setup and an agentic RAG approach?', 'Why would a retrieval system need multiple steps instead of just fetching documents once?', 'What should I have set up on my machine before starting the agentic RAG lessons?', 'How does the course suggest preparing the coding environment for the project?', 'In the RAG lesson, what are the core pieces that make up the standard retrieval-and-generation pipeline?']


In [120]:
usage.input_tokens, usage.output_tokens

(261, 102)

In [121]:
# question 1 140

In [122]:
import pandas as pd
ground_truth = pd.read_csv("ground-truth.csv")
df_ground_truth = pd.DataFrame(ground_truth)

In [123]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [124]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [125]:
len(chunks)

295

In [126]:
from minsearch import Index
index = Index(text_fields=["content"], keyword_fields=["filename"])
index.fit(chunks)

def text_search(query, num_results=5):
    return index.search(query, num_results=num_results)

from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["filename"])


def vector_search(query, num_results=5):
    return vindex.search(query, num_results=num_results)

In [127]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [128]:
def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

In [130]:
df = pd.read_csv("ground-truth.csv")
ground_truth = df.to_dict(orient="records")
q = ground_truth[0]["question"]

In [131]:
text_search_results = text_search(q, num_results=5)

In [132]:
text_search_results[0]

{'start': 3000,
 'content': 'we drop it.\n\nBuild a prompt that includes both the question and the context:\n\n```python\nprompt = f"""\nYour task is to answer questions from the course participants\nbased on the provided context.\n\nUse the context to find relevant information and provide accurate\nanswers. If the answer is not found in the context,\nrespond with "I don\'t know."\n\nQuestion:\n{question}\n\nContext:\n{context}\n"""\n```\n\nInstead of sending the raw question to the LLM, we send this prompt:\n\n```python\nanswer = llm(prompt)\nprint(answer)\n```\n\nAfter that, the answer is correct: "Yes, you can still join. If you want to\nreceive a certificate, you need to submit your project while\nsubmissions are still open."\n\nThis is the answer we actually want to give to our students. What we\njust did is nothing but RAG.\n\n## Retrieval plus generation\n\nRAG stands for Retrieval-Augmented Generation. Generation is the LLM\nproducing text, and retrieval is search. We retrieve 

In [ ]:
# question 2 01-agentic-rag/lessons/03-rag.md